This notebook is used to take a geotif in geographic coordinates (lat/lon) and convert it to radar coordinates, which we call "ungeocoding." In this case, we take the CDL and convert it to the radar coordinates of our coregistered stack of SLCs. 
<br>
<br>
Use the "earthscope_insar" conda environment as the Python kernel.
***

In [1]:
# import necessary packages
import argparse
import isce
import os
from osgeo import gdal
from osgeo import gdal_array
import numpy as np
import xml.etree.ElementTree as ET
import rasterio
import rioxarray
import glob

In [2]:
# define your geometry file names
# the following files need to be the same size as your output grid.
latfile  = 'lat.rdr.full'
lonfile  = 'lon.rdr.full'

In [ ]:
# define the resample method
resampMethod = 'near'

# give the paths to the directories we need

# this is where the CDL sits
indir = './' # present working directory in this case

# this is where we want to put the "ungeocoded" CDL
outdir = '../3_Making_figures/Figure_7-12_CDL_map_and_focus_areas/'

# make this folder to hold some temporary files, I think. 
cropradardir = indir+'radarfullres/'
os.mkdir(cropradardir) # needs to exist and be empty

# the geotiff in geographic coordinates
infile = 'CDL_2023_clip_20240705112914_227417551.tif'

os.chdir(indir) # change to cropscape dir

In [4]:
# figure out bounds of lat/lonfile and crop input file
lf = gdal.Open(latfile)
stats = lf.GetRasterBand(1).GetStatistics(0,1)
sarMinLat = stats[0]
sarMaxLat = stats[1]#SAME AS ABOVEEE
lf=None

# figure out bounds of lat/lonfile and crop input file
lf = gdal.Open(lonfile)
stats = lf.GetRasterBand(1).GetStatistics(0,1)
sarMinLon = stats[0]
sarMaxLon = stats[1] #SAME AS ABOVE BUT FOR X
lf=None

# get pixel spacing on infile
lf    = gdal.Open(infile) #can't read this in either
stats = lf.GetGeoTransform()
dLon  = stats[1] #bleh.x[1] - bleh.x[0]
dLat  = -stats[5] #bleh.y[1] - bleh.y[0]
band = lf.GetRasterBand(1)
arr = band.ReadAsArray()
dtype = gdal.GetDataTypeName(band.DataType)
print(dtype) # this gives me "Byte"
lf   = None

Byte


In [ ]:
# Crop file to one pixel wider than extent of radar coordinates data

cropfile = outdir+os.path.basename(infile)
cmd = 'gdalwarp -te '+str(sarMinLon-dLon)+' '+str(sarMinLat-dLat)+' '+str(sarMaxLon+dLon)+' '+str(sarMaxLat+dLat)+' '+infile+' '+cropfile
os.system(cmd)
# build a vrt file for our tiff
cmd='gdalbuildvrt '+cropfile+'.vrt '+cropfile
os.system(cmd)

Copying color table from CDL_2023_clip_20240705112914_227417551.tif to new file.
Creating output file that is 1249P x 771L.
Processing CDL_2023_clip_20240705112914_227417551.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


0

In [7]:
def writeVRT(infile, latFile, lonFile):
#This function is modified from isce2gis.py
            tree = ET.parse(infile + '.vrt')
            root = tree.getroot()

            meta = ET.SubElement(root, 'metadata')
            meta.attrib['domain'] = "GEOLOCATION"
            meta.tail = '\n'
            meta.text = '\n    '


            rdict = { 'Y_DATASET' : latFile,
                      'X_DATASET' : lonFile,
                      'X_BAND' : "1",
                      'Y_BAND' : "1",
                      'PIXEL_OFFSET': "0",
                      'LINE_OFFSET' : "0",
                      'LINE_STEP' : "1",
                      'PIXEL_STEP' : "1" }

            for key, val in rdict.items():
                data = ET.SubElement(meta, 'mdi')
                data.text = val
                data.attrib['key'] = key
                data.tail = '\n    '

            data.tail = '\n'
            tree.write(infile + '.vrt')

In [8]:
##get number of rows and columns in radar coords file (lat)
raster = gdal.Open(latfile)
radarnx=raster.RasterXSize #bleh.x.shape
radarny=raster.RasterYSize #bleh.y.shape
cols,rows=np.meshgrid(np.arange(1,radarnx+1),np.arange(radarny,0,-1))

#make new files of columns and rows, in radar coordinates
colfile = 'colsfr.r4'
rowfile = 'rowsfr.r4'
gcolfile = 'geo_colsfr.r4'
growfile = 'geo_rowsfr.r4'
driver=gdal.GetDriverByName('ISCE')

In [9]:
colds = driver.Create(colfile,radarnx,radarny,1,gdal.GDT_Float32)
colds.GetRasterBand(1).WriteArray(cols)
colds=None

In [10]:
colds = driver.Create(rowfile,radarnx,radarny,1,gdal.GDT_Float32)
colds.GetRasterBand(1).WriteArray(rows)
colds=None

In [11]:
cmd = 'fixImageXml.py -i '+colfile+' -f'
os.system(cmd)
cmd = 'fixImageXml.py -i '+rowfile+' -f'
os.system(cmd)

fixing xml file path for file: colsfr.r4
fixing xml file path for file: rowsfr.r4


0

In [12]:
writeVRT(colfile, latfile, lonfile)
writeVRT(rowfile, latfile, lonfile)

In [13]:
##get number of rows and columns + min/max range of input file
cropfile = outdir+os.path.basename(infile)
raster = gdal.Open(cropfile)
geonx=raster.RasterXSize
geony=raster.RasterYSize

minLon = raster.GetGeoTransform()[0]
deltaLon = raster.GetGeoTransform()[1]
maxLat = raster.GetGeoTransform()[3]
deltaLat = raster.GetGeoTransform()[5]
minLat = maxLat + geony*deltaLat
maxLon = minLon + geonx*deltaLon
WSEN = str(minLon)+' '+str(minLat)+' '+str(maxLon)+' '+str(maxLat)

In [14]:
#geocode row and col file
cmd = 'gdalwarp -of ISCE -overwrite -geoloc  -te '+WSEN+' -tr '+str(deltaLon)+' '+str(deltaLat)+' -srcnodata 0 -dstnodata 0  -wt float32 -r bilinear ' +rowfile +'.vrt ' + growfile
print(cmd)
os.system(cmd)
cmd = 'gdalwarp -of ISCE -overwrite -geoloc  -te '+WSEN+' -tr '+str(deltaLon)+' '+str(deltaLat)+' -srcnodata 0 -dstnodata 0  -wt float32 -r bilinear ' +colfile +'.vrt ' + gcolfile
print(cmd)
os.system(cmd)
cmd = 'gdalbuildvrt '+gcolfile+'.vrt '+gcolfile
os.system(cmd)
cmd = 'gdalbuildvrt '+growfile+'.vrt '+growfile
os.system(cmd)

gdalwarp -of ISCE -overwrite -geoloc  -te -115.87604044228875 32.742998367860594 -115.46904874333124 32.994148032385404 -tr 0.00032585404239992953 -0.00032574534957822325 -srcnodata 0 -dstnodata 0  -wt float32 -r bilinear rowsfr.r4.vrt geo_rowsfr.r4
Creating output file that is 1249P x 771L.
Processing rowsfr.r4.vrt [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
gdalwarp -of ISCE -overwrite -geoloc  -te -115.87604044228875 32.742998367860594 -115.46904874333124 32.994148032385404 -tr 0.00032585404239992953 -0.00032574534957822325 -srcnodata 0 -dstnodata 0  -wt float32 -r bilinear colsfr.r4.vrt geo_colsfr.r4
Creating output file that is 1249P x 771L.
Processing colsfr.r4.vrt [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


0

In [15]:
###set any geocoding info
WSEN = '0.5 0.5 '+str(radarnx+0.5)+' '+str(radarny+0.5)

cropfile = outdir+os.path.basename(infile)
outfile  = cropradardir+os.path.basename(infile)
writeVRT(cropfile, growfile, gcolfile)
cmd = 'gdalwarp -overwrite -geoloc -of ISCE -te '+ WSEN + ' -tr 1 1 -srcnodata 0 -dstnodata 0 -wt '+dtype+' -r ' + resampMethod + ' ' + cropfile +'.vrt ' +outfile
os.system(cmd)
cmd = 'gdalbuildvrt '+outfile+'.vrt '+outfile
os.system(cmd)

Copying color table from ../3_Making_figures/Figure_7-12_CDL_map_and_focus_areas/CDL_2023_clip_20240705112914_227417551.tif.vrt to new file.
Creating output file that is 9000P x 1575L.
Processing ../3_Making_figures/Figure_7-12_CDL_map_and_focus_areas/CDL_2023_clip_20240705112914_227417551.tif.vrt [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


0

In [ ]:
# cropped, ungeocoded CDL tiff now made. In this case, it is called "CDL_2023_clip_20240705112914_227417551.tif"
# and it's in the "/Figure_7-12_CDL_map_and_focus_areas/"" folder
# and it should be exactly the same as the file "CDL_2023_cropped.tif" which just has a simplified name.